# Drought analysis


In [5]:
# general
from tqdm import tqdm
from pathlib import Path
import pandas as pd

# geospatial
import geopandas as gpd
import xarray as xr
from src.utils import load_config
from src.droughts import find_droughts, drought_area_timeseries

cfg = load_config()

## Data

In [6]:
# Andean-GC
ids = sum([Path(f"modelling/{s}.txt").read_text().splitlines() for s in ["basins_test", "basins_training", "basins_validation"]], [])

## simulated runoff for Andean-GC
runoff_andeangc_sim = xr.open_dataset("data/hydro/runoff_sim_andeangc.nc").q_mm_sim
runoff_andeangc_sim = runoff_andeangc_sim.sel(gauge_id=ids)

## observed runoff for Andean-GC
metadata_andeangc   = pd.read_csv(Path(cfg["dir"]["andeangc"]) / "dataset/AndeanGC_metadata.csv", index_col=0)
runoff_andeangc_obs = pd.read_csv(Path(cfg["dir"]["andeangc"]) / "dataset/AndeanGC_data_1950_2024.csv", index_col=0, parse_dates=True)
runoff_andeangc_obs = runoff_andeangc_obs.loc[cfg["time"]["ref_start"]:cfg["time"]["ref_end"]]
runoff_andeangc_obs = (runoff_andeangc_obs*1000*86400) / (metadata_andeangc.basin_area*1e6)
runoff_andeangc_obs = xr.DataArray(
    runoff_andeangc_obs.values,
    coords={"gauge_id": runoff_andeangc_obs.columns, "date": runoff_andeangc_obs.index},
    dims=("date", "gauge_id" ),
    name="q_mm")

# remove NaNs in either observed or simulated
runoff_andeangc_obs = runoff_andeangc_obs.where(runoff_andeangc_sim.notnull(), drop=True)
runoff_andeangc_sim = runoff_andeangc_sim.where(runoff_andeangc_obs.notnull(), drop=True)

In [7]:
# hydroATLAS
metadata_hydroatlas = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg").set_index("gauge_id")

runoff_hydroatlas_sim = xr.open_dataset("data/hydro/runoff_sim_hydroatlas_hist.nc").dropna(dim="date").q_mm_sim
runoff_hydroatlas_whatif = xr.open_dataset("data/hydro/runoff_sim_hydroatlas_whatif.nc").dropna(dim="date").q_mm_sim
runoff_hydroatlas_whatif = runoff_hydroatlas_whatif.groupby('whatif_scenario').mean() # multi model mean

## Streamflow droughts comparison for AndeanGC

In [8]:
# Calculate droughts for observed and simulated data
droughts_obs = find_droughts(runoff_andeangc_obs, 
                             threshold_quantile=cfg["hydro"]["drought_threshold"], 
                             threshold_df=None, 
                             pooling_window=cfg["hydro"]["drought_pooling_window"], 
                             min_duration=cfg["hydro"]["drought_min_duration"], 
                             seasonal_threshold=True,
                             suffix="_obs")

droughts_sim = find_droughts(runoff_andeangc_sim, 
                             threshold_quantile=cfg["hydro"]["drought_threshold"], 
                             threshold_df=None, 
                             pooling_window=cfg["hydro"]["drought_pooling_window"], 
                             min_duration=cfg["hydro"]["drought_min_duration"], 
                             seasonal_threshold=True,
                             suffix="_sim")

# Combine and save results
df = pd.concat([droughts_obs, droughts_sim], axis=1)
df.to_csv("data/hydro/performance_lowflows.csv")


## Streamflow droughts for all HydroATLAS basins (+ drought extent)

In [ ]:
# Calculate reference droughts
droughts_ref = find_droughts(runoff_hydroatlas_sim, cfg["hydro"]["drought_threshold"], 
                             suffix="",
                             pooling_window=cfg["hydro"]["drought_pooling_window"],
                             min_duration=cfg["hydro"]["drought_min_duration"])
droughts_ts = drought_area_timeseries(runoff_hydroatlas_sim,  metadata_hydroatlas, 
                                      cfg["hydro"]["drought_threshold"], threshold_df=None,
                                      pooling_window=cfg["hydro"]["drought_pooling_window"],
                                      min_duration=cfg["hydro"]["drought_min_duration"])
droughts_ts = pd.concat([droughts_ts], keys=['historical'], names=['scenario'])

# Initialize results list
results = []

# Process each warming scenario
for sce in tqdm(['15C', '28C', '40C']): 
    
    # Select scenario
    runoff_whatif_sce = runoff_hydroatlas_whatif.sel(whatif_scenario=sce, drop=True)

    # Find droughts for scenario
    drought_whatif = find_droughts(runoff_whatif_sce, cfg["hydro"]["drought_threshold"], 
                                   suffix="", threshold_df=runoff_hydroatlas_sim,
                                   pooling_window=cfg["hydro"]["drought_pooling_window"],
                                   min_duration=cfg["hydro"]["drought_min_duration"])

    # Calculate drought area timeseries
    drought_ts_whatif = drought_area_timeseries(runoff_whatif_sce, metadata_hydroatlas, 
                                                cfg["hydro"]["drought_threshold"], 
                                                threshold_df=runoff_hydroatlas_sim,
                                                pooling_window=cfg["hydro"]["drought_pooling_window"],
                                                min_duration=cfg["hydro"]["drought_min_duration"])
    drought_ts_whatif = pd.concat([drought_ts_whatif], keys=[sce], names=['scenario'])
    droughts_ts = pd.concat([droughts_ts, drought_ts_whatif])
    
    # Calculate percent changes for most metrics
    percent_metrics = ['Q90', 'annual_min_value', 'min_30day', 'severity', 'total_duration', 'intensity']
    for metric in percent_metrics:
        drought_whatif[metric] = ((drought_whatif[metric] - droughts_ref[metric]) / droughts_ref[metric]) * 100

    # Calculate absolute differences for specific metrics
    drought_whatif['doy_min'] = drought_whatif['doy_min'] - droughts_ref['doy_min']
    drought_whatif['events_per_year'] = drought_whatif['events_per_year'] - droughts_ref['events_per_year']
    
    # Add scenario identifier
    drought_whatif["scenario"] = sce
    results.append(drought_whatif)

# Combine all scenarios and save
df = pd.concat(results, axis=0)
df.to_csv("data/hydro/droughts_hydroatlas_whatif.csv")
droughts_ts.index.names = ["scenario", "date"]
droughts_ts.to_csv("data/hydro/droughts_hydroatlas_area_ts.csv")
